In [ ]:
%pip install torch transformers datasets pandas numpy
%pip install datasets transformers torch

In [2]:
%pip install ntlk

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement ntlk (from versions: none)
ERROR: No matching distribution found for ntlk

[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [94]:
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer
import torch
import pandas as pd

In [4]:
import nltk
nltk.download('punkt')  # Needed for word_tokenize
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\youst\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [92]:
dataset = load_dataset("rajpurkar/squad")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [95]:
train_data = dataset['train']
validation_data = dataset['validation']

# Convert the tensors to lists for each field.
train_contexts = train_data['context']
train_questions = train_data['question']
train_answers = train_data['answers']

validation_contexts = validation_data['context']
validation_questions = validation_data['question']
validation_answers = validation_data['answers']

# Combine the fields into a DataFrame
train_df = pd.DataFrame({
    'context': train_contexts,
    'question': train_questions,
    'answer': [ans['text'][0] for ans in train_answers],  # Extract the first answer
    'answer_start': [ans['answer_start'][0] for ans in train_answers]  # Extract answer start
})

validation_df = pd.DataFrame({
    'context': validation_contexts,
    'question': validation_questions,
    'answer': [ans['text'][0] for ans in validation_answers],
    'answer_start': [ans['answer_start'][0] for ans in validation_answers]
})

# Step 1: Compute the length of each context (in terms of words)
train_df['context_length'] = train_df['context'].apply(lambda x: len(x.split()))  # Length in terms of words
validation_df['context_length'] = validation_df['context'].apply(lambda x: len(x.split()))

# Step 2: Sort by the length of the context
train_df_sorted = train_df.sort_values(by='context_length')
validation_df_sorted = validation_df.sort_values(by='context_length')

# Step 3: Select the shortest 5000 contexts from train and 2000 from validation
train_selected = train_df_sorted.head(5000)
validation_selected = validation_df_sorted.head(2000)

# Step 4: Convert back to Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_selected[['context', 'question', 'answer', 'answer_start']])
validation_dataset = Dataset.from_pandas(validation_selected[['context', 'question', 'answer', 'answer_start']])

# Step 5: Confirm the datasets
print("Train Dataset")
print(train_dataset)

print("Validation Dataset")
print(validation_dataset)

Train Dataset
Dataset({
    features: ['context', 'question', 'answer', 'answer_start', '__index_level_0__'],
    num_rows: 5000
})
Validation Dataset
Dataset({
    features: ['context', 'question', 'answer', 'answer_start', '__index_level_0__'],
    num_rows: 2000
})


In [ ]:
from math import floor

# First 5 entries
print("First 5 entries:")
for i in range(5):
    print(train_dataset[i])
    print("-" * 80)

# Middle 5 entries
print("Middle 5 entries:")
middle_start = floor(len(train_dataset) / 2) - 2
for i in range(middle_start, middle_start + 5):
    print(train_dataset[i])
    print("-" * 80)

# Last 5 entries
print("Last 5 entries:")
for i in range(len(train_dataset) - 5, len(train_dataset)):
    print(train_dataset[i])
    print("-" * 80)

First 5 entries:
{'context': 'Internet services typically provided by ISPs include Internet access, Internet transit, domain name registration, web hosting, Usenet service, and colocation.', 'question': 'What type of organization provides internet access?', 'answer': 'ISPs', 'answer_start': 40, '__index_level_0__': 13934}
--------------------------------------------------------------------------------
{'context': 'Only a few contemporary societies are classified as hunter-gatherers, and many supplement their foraging activity with horticulture and/or keeping animals.', 'question': 'How many groups of modern hunter-gatherers are there?', 'answer': 'Only a few', 'answer_start': 0, '__index_level_0__': 12994}
--------------------------------------------------------------------------------
{'context': 'Only a few contemporary societies are classified as hunter-gatherers, and many supplement their foraging activity with horticulture and/or keeping animals.', 'question': 'What do modern hunt

In [98]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

df = train_dataset.to_pandas()

df['input'] = "Context: " + df['context'] + " Question: " + df['question']

# Adjust answer_start to account for the added "Context: "
prefix_len = len("Context: ")

# Recalculate answer_start to align with full input
df['adjusted_answer_start'] = df['answer_start'] + prefix_len

# Tokenize inputs and compute word-level start and end positions
tokenized_inputs = []
start_word_indices = []
end_word_indices = []

for i, row in df.iterrows():
    input_text = row['input']
    answer_text = row['answer']
    answer_char_start = row['adjusted_answer_start']
    
    # Tokenize input
    tokens = word_tokenize(input_text)
    tokenized_inputs.append(tokens)
    
    # Track character positions of each token
    char_positions = []
    running_char_pos = 0
    for token in tokens:
        char_index = input_text.find(token, running_char_pos)
        char_positions.append((token, char_index))
        running_char_pos = char_index + len(token)

    # Find the token index where the answer starts
    start_token_idx = None
    for idx, (tok, char_pos) in enumerate(char_positions):
        if char_pos >= answer_char_start:
            start_token_idx = idx
            break

    if start_token_idx is None:
        start_token_idx = len(tokens) - 1  # fallback
    
    # Tokenize answer separately to get answer length in words
    answer_tokens = word_tokenize(answer_text)
    end_token_idx = start_token_idx + len(answer_tokens) - 1

    start_word_indices.append(start_token_idx)
    end_word_indices.append(end_token_idx)

# Add to DataFrame
df['input_tokens'] = tokenized_inputs
df['answer_start_word'] = start_word_indices
df['answer_end_word'] = end_word_indices

train_df = df[['input', 'input_tokens', 'answer', 'answer_start_word', 'answer_end_word']]

print(train_df.head())


                                               input  \
0  Context: Internet services typically provided ...   
1  Context: Only a few contemporary societies are...   
2  Context: Only a few contemporary societies are...   
3  Context: Only a few contemporary societies are...   
4  Context: Only a few contemporary societies are...   

                                        input_tokens  \
0  [Context, :, Internet, services, typically, pr...   
1  [Context, :, Only, a, few, contemporary, socie...   
2  [Context, :, Only, a, few, contemporary, socie...   
3  [Context, :, Only, a, few, contemporary, socie...   
4  [Context, :, Only, a, few, contemporary, socie...   

                                answer  answer_start_word  answer_end_word  
0                                 ISPs                  7                7  
1                           Only a few                  2                4  
2  horticulture and/or keeping animals                 19               22  
3                 

In [99]:
import numpy as np
from nltk.tokenize import word_tokenize
import torch
from collections import Counter
from tqdm import tqdm

In [102]:
glove_path = "glove.6B.100d.txt"
embedding_dim = 100
glove_embeddings = {}

with open(glove_path, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Loading GloVe"):
        parts = line.strip().split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove_embeddings[word] = vector

def get_embedding_sequence(tokens, embeddings_dict, embedding_dim):
    vectors = []
    for token in tokens:
        if token in embeddings_dict:
            vectors.append(embeddings_dict[token])
        else:
            # Use a zero vector if word not in GloVe
            vectors.append(np.zeros(embedding_dim))
    return np.array(vectors)

train_df['embedding_sequence'] = train_df['input_tokens'].apply(
    lambda tokens: get_embedding_sequence(tokens, glove_embeddings, embedding_dim)
)

print(train_df['embedding_sequence'].iloc[0].shape)  # (sequence_len, 100)


Loading GloVe: 400000it [00:13, 29046.81it/s]


(38, 100)


C:\Users\youst\AppData\Local\Temp\ipykernel_13652\3067312080.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['embedding_sequence'] = train_df['input_tokens'].apply(


In [104]:
MAX_LEN = 100
EMBED_DIM = 100

padded_embeddings = []
start_positions = []
end_positions = []
attention_masks = []

for _, row in train_df.iterrows():
    tokens = row['input_tokens']
    start_idx = row['answer_start_word']
    end_idx = row['answer_end_word']

    # Truncate if too long
    if len(tokens) > MAX_LEN:
        tokens = tokens[:MAX_LEN]
        if start_idx >= MAX_LEN or end_idx >= MAX_LEN:
            continue  # Skip samples where answer is outside the range
        start_idx = min(start_idx, MAX_LEN - 1)
        end_idx = min(end_idx, MAX_LEN - 1)

    # Embed the tokens using GloVe
    embedded = [glove_embeddings.get(token, np.zeros(EMBED_DIM)) for token in tokens]

    # Padding
    pad_len = MAX_LEN - len(embedded)
    if pad_len > 0:
        embedded += [np.zeros(EMBED_DIM)] * pad_len

    mask = [1] * len(tokens) + [0] * pad_len

    padded_embeddings.append(embedded)
    attention_masks.append(mask)
    start_positions.append(start_idx)
    end_positions.append(end_idx)

# Convert to tensors
inputs_tensor = torch.tensor(padded_embeddings, dtype=torch.float32)
start_tensor = torch.tensor(start_positions, dtype=torch.long)
end_tensor = torch.tensor(end_positions, dtype=torch.long)
mask_tensor = torch.tensor(attention_masks, dtype=torch.float32)


C:\Users\youst\AppData\Local\Temp\ipykernel_13652\3861726047.py:38: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:257.)
  inputs_tensor = torch.tensor(padded_embeddings, dtype=torch.float32)


In [116]:
from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(inputs_tensor, start_tensor, end_tensor)
train_loader = DataLoader(train_data, batch_size=8, shuffle=True)

In [140]:
import torch.nn as nn

class QA_Model(nn.Module):
    def __init__(self, embedding_dim=100, hidden_dim=256, dropout=0.2):
        super(QA_Model, self).__init__()
        
        self.hidden_dim = hidden_dim

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(hidden_dim, 2)  # Output: start & end logits per token

    def forward(self, x, attention_mask=None):
        lstm_out, _ = self.lstm(x)  # (batch_size, seq_len, hidden_dim*2)

        x = self.fc1(lstm_out)      # (batch_size, seq_len, hidden_dim)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)        # (batch_size, seq_len, 2)

        start_logits, end_logits = logits.split(1, dim=-1)
        start_logits = start_logits.squeeze(-1)
        end_logits = end_logits.squeeze(-1)

        if attention_mask is not None:
            start_logits = start_logits * attention_mask
            end_logits = end_logits * attention_mask

        return start_logits, end_logits


In [141]:
# Hyperparameters
EMBEDDING_DIM = 100
HIDDEN_DIM = 256
DROPOUT = 0.2

model = QA_Model(
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [142]:
from tqdm import tqdm
from torch.nn import CrossEntropyLoss

def train_model(model, train_loader, optimizer, num_epochs=5, device="cpu"):
    model.to(device)
    criterion = CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        correct_start = 0
        correct_end = 0
        total = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch in loop:
            embeddings, start_positions, end_positions = batch
            embeddings = embeddings.to(device)
            start_positions = start_positions.to(device)
            end_positions = end_positions.to(device)

            optimizer.zero_grad()

            start_logits, end_logits = model(embeddings)

            loss_start = criterion(start_logits, start_positions)
            loss_end = criterion(end_logits, end_positions)
            loss = loss_start + loss_end
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            pred_start = torch.argmax(start_logits, dim=1)
            pred_end = torch.argmax(end_logits, dim=1)
            correct_start += (pred_start == start_positions).sum().item()
            correct_end += (pred_end == end_positions).sum().item()
            total += start_positions.size(0)

            loop.set_postfix({
                'Loss': loss.item(),
                'Acc Start': f"{(correct_start / total) * 100:.2f}%",
                'Acc End': f"{(correct_end / total) * 100:.2f}%"
            })

        avg_loss = total_loss / len(train_loader)
        avg_acc_start = correct_start / total
        avg_acc_end = correct_end / total

        print(f"Epoch {epoch+1}/{num_epochs}")
        print(f"Average Loss: {avg_loss:.4f}")
        print(f"Start Accuracy: {avg_acc_start * 100:.2f}%")
        print(f"End Accuracy: {avg_acc_end * 100:.2f}%\n")


In [143]:
train_model(model, train_loader, optimizer, num_epochs=20, device='cpu') 

Epoch 1/20: 100%|██████████| 625/625 [01:30<00:00,  6.87it/s, Loss=5.56, Acc Start=7.86%, Acc End=8.86%]


Epoch 1/20
Average Loss: 7.0432
Start Accuracy: 7.86%
End Accuracy: 8.86%



Epoch 2/20: 100%|██████████| 625/625 [01:15<00:00,  8.32it/s, Loss=5.35, Acc Start=12.04%, Acc End=13.94%]


Epoch 2/20
Average Loss: 6.2068
Start Accuracy: 12.04%
End Accuracy: 13.94%



Epoch 3/20: 100%|██████████| 625/625 [01:16<00:00,  8.18it/s, Loss=5.63, Acc Start=12.96%, Acc End=14.34%]


...

Epoch 18/20
Average Loss: 5.1840
Start Accuracy: 18.78%
End Accuracy: 20.34%



Epoch 19/20: 100%|██████████| 625/625 [01:15<00:00,  8.31it/s, Loss=4.75, Acc Start=19.48%, Acc End=21.18%]


Epoch 19/20
Average Loss: 5.1160
Start Accuracy: 19.48%
End Accuracy: 21.18%



Epoch 20/20: 100%|██████████| 625/625 [01:16<00:00,  8.22it/s, Loss=3.98, Acc Start=19.34%, Acc End=21.08%]

Epoch 20/20
Average Loss: 5.0363
Start Accuracy: 19.34%
End Accuracy: 21.08%

